In [9]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import pandas as pd
import json

from scripts import analyser
from scripts import runner

In [11]:
def get_dataframe_metadata(df):

    metadata = {
        "row_count": len(df),
        "column_count": len(df.columns),
        "columns": {}
    }

    for column in df.columns:

        metadata["columns"][column] = {
            "dtype": str(df[column].dtype),
            "missing_count": int(df[column].isna().sum()),
            "unique_count": int(df[column].nunique()),
            "sample_values": (
                df[column]
                .dropna()
                .head(5)
                .tolist()
            )
        }

    return metadata

In [12]:
df = pd.read_csv(r"C:\Users\shubh\Desktop\Auto EDA\datasets\superhero_movie_sales_clean_500.csv")

metadata = get_dataframe_metadata(df)

df.head()


,order_id,order_date,universe,movie_title,movie_type,product_type,sales_channel,region,customer_segment,units_sold,unit_price_usd,discount_pct,gross_sales_usd,discount_amount_usd,net_sales_usd,rental_days
0,ORD00010,2024-01-03,DC,The Batman,Solo Hero,Digital Purchase,HeroShelf,Europe,Casual Viewer,1,13.49,15,13.49,2.02,11.47,0
1,ORD00445,2024-01-03,Marvel,Thor: Love and Thunder,Solo Hero,Digital Purchase,HeroShelf,Europe,Fan,2,14.24,10,28.48,2.85,25.63,0
2,ORD00094,2024-01-06,Marvel,Shang-Chi and the Legend of the Ten Rings,Solo Hero,Blu-ray,HeroShelf,North America,Casual Viewer,2,27.49,10,54.98,5.50,49.48,0
3,ORD00318,2024-01-09,DC,The Dark Knight,Solo Hero,Blu-ray,Online Marketplace,North America,Fan,1,27.49,5,27.49,1.37,26.12,0
4,ORD00017,2024-01-10,DC,Shazam! Fury of the Gods,Solo Hero,Digital Rental,CineNow,North America,Fan,2,5.22,5,10.44,0.52,9.92,5


In [13]:
basic_eda_steps = analyser.get_analysis_recommendations_basic(df)

print(basic_eda_steps)

Based on your dataset structure (500 rows, 16 columns) and sample values, here is a structured exploratory data analysis (EDA) plan. It is tailored to uncover financial performance, operational tendencies, and consumer behavior patterns without relying on identifiers or repetitive steps.

---

### 1. Descriptive Statistics of Financial and Quantitative Metrics
* **Analysis Objective:** Summarize central tendencies, spread, and boundaries for numerical measures to understand baseline pricing, sales volumes, discounting strategies, and rental durations.
* **Exact Column Names & Data Types:** 
  - `units_sold` (int64)
  - `unit_price_usd` (float64)
  - `discount_pct` (int64)
  - `gross_sales_usd` (float64)
  - `discount_amount_usd` (float64)
  - `net_sales_usd` (float64)
  - `rental_days` (int64)
* **Operations & Functions:** Use pandas `.describe()` method, supplemented by `.agg()` to calculate specific metrics including standard deviation, interquartile range (IQR), and variance.
* **Re

In [14]:
analysis_functions_json = analyser.generate_analysis_functions(basic_eda_steps, df)

print(analysis_functions_json)

{
    "functions": [
        {
            "name": "summarize_financial_metrics",
            "columns": [
                "units_sold",
                "unit_price_usd",
                "discount_pct",
                "gross_sales_usd",
                "discount_amount_usd",
                "net_sales_usd",
                "rental_days"
            ],
            "action": "descriptive_statistics",
            "reason": "Summarize central tendencies, spread, and boundaries for numerical measures to understand baseline pricing, sales volumes, discounting strategies, and rental durations.",
            "requires_review": false,
            "code": "def summarize_financial_metrics(df):\n    num_cols = [\n        'units_sold',\n        'unit_price_usd',\n        'discount_pct',\n        'gross_sales_usd',\n        'discount_amount_usd',\n        'net_sales_usd',\n        'rental_days'\n    ]\n    subset = df[num_cols].copy()\n    desc = subset.describe(percentiles=[0.25, 0.50, 0.75])\n   

In [15]:
output = runner.execute_eda_steps(analysis_functions_json, df, metadata)

print(output)

[SUCCESS] summarize_financial_metrics ran successfully and output is stored.
[SUCCESS] analyze_categorical_distributions ran successfully and output is stored.
[SUCCESS] aggregate_product_channel_financials ran successfully and output is stored.
[SUCCESS] pivot_regional_customer_revenue ran successfully and output is stored.
[SUCCESS] calculate_numerical_correlations ran successfully and output is stored.
[SUCCESS] analyze_temporal_sales_trends ran successfully and output is stored.
[SUCCESS] investigate_anomalies_and_extremes ran successfully and output is stored.
Analysis process completed.
[{'name': 'summarize_financial_metrics', 'actions': 'descriptive_statistics', 'reason': 'Summarize central tendencies, spread, and boundaries for numerical measures to understand baseline pricing, sales volumes, discounting strategies, and rental durations.', 'output':        units_sold  unit_price_usd  discount_pct  gross_sales_usd  \
count  500.000000      500.000000    500.000000        500.000

In [16]:
adv_recommendations = analyser.get_analysis_recommendations_advanced(output, df, basic_eda_steps)

print(adv_recommendations)

## Key observations
- **Step 1 (Financial Metrics):** Showed that `net_sales_usd` (mean: 21.46, max: 109.96) is heavily driven by `gross_sales_usd` and discounting, with significant right-skew indicated by maximum values far exceeding 75th percentiles.
- **Step 2 (Categorical Distributions):** Revealed Marvel dominates the catalog (60%), Solo Heroes represent ~71% of movies, and Digital channels (Rental/Purchase) account for over 60% of product types, with North America leading geographic distribution (44.2%).
- **Step 3 (Product/Channel Financials):** Highlighted that StreamFlix drives the highest total net sales for Digital Purchases ($1309.52) and Rentals ($645.25), while Merchandise performs exceptionally well on Online Marketplaces ($1220.97).
- **Step 4 (Regional/Customer Revenue):** Uncovered that North America is the top revenue region across all segments, led heavily by Casual Users ($2036.78) and Fans ($1464.69).
- **Step 5 (Correlations):** Demonstrated near-perfect positive

In [17]:
adv_functions = analyser.generate_analysis_functions(adv_recommendations, df)

print(adv_functions)

{
    "functions": [
        {
            "name": "analyze_holiday_seasonality",
            "columns": ["order_date", "net_sales_usd", "units_sold", "product_type"],
            "action": "monthly_pivot_analysis",
            "reason": "Quantifies December sales spikes and compares baseline performance across years and product types.",
            "requires_review": false,
            "code": "def analyze_holiday_seasonality(df):\n    d = df.copy()\n    d['order_date'] = pd.to_datetime(d['order_date'])\n    d['year'] = d['order_date'].dt.year\n    d['month'] = d['order_date'].dt.month\n    pivot = d.groupby(['year', 'month', 'product_type']).agg({\n        'net_sales_usd': ['sum', 'mean'],\n        'units_sold': 'sum'\n    }).reset_index()\n    return pivot"
        },
        {
            "name": "analyze_discount_profitability",
            "columns": ["customer_segment", "discount_pct", "net_sales_usd", "discount_amount_usd", "units_sold"],
            "action": "discount_segment

In [18]:
output_adv = runner.execute_eda_steps(adv_functions, df, metadata)

[SUCCESS] analyze_holiday_seasonality ran successfully and output is stored.
[SUCCESS] analyze_discount_profitability ran successfully and output is stored.
[SUCCESS] profile_outlier_transactions ran successfully and output is stored.
[SUCCESS] analyze_rental_pricing_structure ran successfully and output is stored.
[SUCCESS] analyze_cross_channel_market_share ran successfully and output is stored.
Analysis process completed.


In [20]:
final_insights = analyser.final_insights_summariser(basic_eda_steps, adv_recommendations,output, output_adv)

print(final_insights)

1.  Net sales show a wide range, with a few high-value transactions significantly boosting the average.
    Evidence: The average net sales per transaction is $21.46, but the maximum recorded net sales is $109.96, which is more than four times the 75th percentile value of $26.12.

2.  Most transactions involve a small number of units, indicating a predominantly retail-focused sales model rather than bulk purchases.
    Evidence: The average units sold per transaction is 1.696, with 75% of transactions involving 2 units or fewer, and a maximum of 5 units sold in a single transaction.

3.  Discounts are a common practice, with a typical range, though some transactions occur without any discount.
    Evidence: The average discount percentage is 11.48%, with discounts ranging from 0% to 30%.

4.  Marvel content and Solo Hero movie types are the most prevalent in the catalog and transactions.
    Evidence: Marvel accounts for 60% of the universe, and Solo Hero movie types represent 70.8% of